- 4a Energy slicing
- 4b Slice fitting
- 4c Neutron channel bounds
- 4d Count rate and dwell time

### Imports/Constants

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    TypeVar,
    Any,
    Literal,
    TypeGuard
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import mpl_toolkits.mplot3d.art3d as art3d
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn,
    EnergyColumn,
    get_df_col
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.slice_fitting import (
#     get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.neutron_classification import classify
from data_processing import processing as proc
from data_processing import types as proc_types
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.processing.calibration import Detector, DetectorCalibrationParams
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: NeutronStrategyFactory,
    window_type: WindowType,
    loading: bool,
    settings: NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def get_psd_adc_histogram(
    df: pd.DataFrame,
    adc_width: float = 420,
    adc_bins: np.ndarray | None = None,
    psd_bin_count: int = 100,
    psd_min: float = 0.0,
    psd_max: float = 0.5
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = get_df_col(df, DetectorDataframeColumn.ENERGY)
    y = get_df_col(df, DetectorDataframeColumn.PSD)

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    if adc_bins is not None:
        x_bins = adc_bins
    else:
        x_bins: np.ndarray = np.linspace(
            0, x.max(), int(x.max() / adc_width) + 1
        )
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    return Z, xe, ye

In [ ]:
# def recalibrate_np(a: np.ndarray):
#     calib_params = DetectorCalibrationParams[detector_code]
#     return (a - calib_params.p2) / (1000 * calib_params.p1)

In [ ]:
def lerp(x: float, p1: tuple[float, float], p2: tuple[float, float]) -> float:
    x1, y1 = p1
    x2, y2 = p2
    m = (y2 - y1) / (x2 - x1)
    return m * (x - x1) + y1


def clamp(x: float, xmin: float, xmax: float) -> float:
    if x < xmin:
        return xmin
    elif x > xmax:
        return xmax
    else:
        return x


def to_unit_interval(x: float, xmin: float, xmax: float) -> float:
    return clamp(lerp(x, (xmin, 0), (xmax, 1)), 0, 1)


ColorHexCode = str  # of format #xxxxxx, can be made into hex value
ColorAlphaHexCode = str  # of format #xxxxxxxx, can be made into hex value
ColorTuple = tuple[float, float, float]
ColorAlphaTuple = tuple[float, float, float, float]
ColorType = ColorHexCode | ColorAlphaHexCode | ColorTuple | ColorAlphaTuple
CT = TypeVar("CT", bound=ColorType)


def is_color_hex_code(x: Any) -> TypeGuard[ColorHexCode]:
    if not isinstance(x, str):
        return False
    pattern_match = re.match(r"#[0-9a-fA-F]{6}$", x)
    return pattern_match is not None


def is_color_alpha_hex_code(x: Any) -> TypeGuard[ColorAlphaHexCode]:
    if not isinstance(x, str):
        return False
    pattern_match = re.match(r"#[0-9a-fA-F]{8}$", x)
    return pattern_match is not None


def is_color_tuple(x: Any) -> TypeGuard[ColorTuple]:
    if not isinstance(x, tuple):
        return False
    if not len(x) == 3:
        return False
    elements_in_limits = [0 <= elem <= 1 for elem in x]
    return all(elements_in_limits)


def is_color_alpha_tuple(x: Any) -> TypeGuard[ColorAlphaTuple]:
    if not isinstance(x, tuple):
        return False
    if not len(x) == 4:
        return False
    elements_in_limits = [0 <= elem <= 1 for elem in x]
    return all(elements_in_limits)


def to_color_float(x: str) -> float:
    try:
        color = int(x, base=16) / 255
    except ValueError:
        raise ValueError(f"{x} is not valid hexadecimal code")
    if color < 0 or color > 1:
        raise ValueError(f"{x} is out of color float bounds (0-1)")
    return color


def to_hex_chars(x: float) -> str:
    if x < 0 or x > 1:
        raise ValueError(f"{x} is out of color float bounds (0-1)")
    x_int = min(255, int(x * 256))
    return f"{x_int:02X}"


def to_color_alpha_tuple(x: ColorType) -> ColorAlphaTuple:
    if is_color_alpha_tuple(x):
        return x
    elif is_color_alpha_hex_code(x):
        r = to_color_float(x[1:3])
        g = to_color_float(x[3:5])
        b = to_color_float(x[5:7])
        a = to_color_float(x[7:9])
        return (r, g, b, a)
    elif is_color_tuple(x):
        r, g, b = x
        return (r, g, b, 1)
    elif is_color_hex_code(x):
        r = to_color_float(x[1:3])
        g = to_color_float(x[3:5])
        b = to_color_float(x[5:7])
        return (r, g, b, 1)
    else:
        raise ValueError(f"{x} could not be converted to ColorAlphaTuple")


def to_color_tuple(x: ColorType) -> ColorTuple:
    if is_color_alpha_tuple(x):
        r, g, b, _ = x
        return (r, g, b)
    elif is_color_alpha_hex_code(x):
        r = to_color_float(x[1:3])
        g = to_color_float(x[3:5])
        b = to_color_float(x[5:7])
        return (r, g, b)
    elif is_color_tuple(x):
        return x
    elif is_color_hex_code(x):
        r = to_color_float(x[1:3])
        g = to_color_float(x[3:5])
        b = to_color_float(x[5:7])
        return (r, g, b)
    else:
        raise ValueError(f"{x} could not be converted to ColorTuple")


def to_color_hex_code(x: ColorType) -> ColorHexCode:
    if is_color_alpha_tuple(x):
        r, g, b, _ = x
        r_hex = to_hex_chars(r)
        g_hex = to_hex_chars(g)
        b_hex = to_hex_chars(b)
        return ("#" + r_hex + g_hex + b_hex).lower()
    elif is_color_alpha_hex_code(x):
        return x[:-2]
    elif is_color_tuple(x):
        r, g, b = x
        r_hex = to_hex_chars(r)
        g_hex = to_hex_chars(g)
        b_hex = to_hex_chars(b)
        return ("#" + r_hex + g_hex + b_hex).lower()
    elif is_color_hex_code(x):
        return x
    else:
        raise ValueError(f"{x} could not be converted to ColorHexCode")


def to_color_alpha_hex_code(x: ColorType) -> ColorAlphaHexCode:
    if is_color_alpha_tuple(x):
        r, g, b, a = x
        r_hex = to_hex_chars(r)
        g_hex = to_hex_chars(g)
        b_hex = to_hex_chars(b)
        a_hex = to_hex_chars(a)
        return ("#" + r_hex + g_hex + b_hex + a_hex).lower()
    elif is_color_alpha_hex_code(x):
        return x
    elif is_color_tuple(x):
        r, g, b = x
        r_hex = to_hex_chars(r)
        g_hex = to_hex_chars(g)
        b_hex = to_hex_chars(b)
        return ("#" + r_hex + g_hex + b_hex + "ff").lower()
    elif is_color_hex_code(x):
        return x+"ff"
    else:
        raise ValueError(f"{x} could not be converted to ColorAlphaHexCode")


def calculate_gradient_color(
    gradient_unit_interval: float,
    color_from: ColorAlphaTuple,
    color_to: ColorAlphaTuple
) -> ColorAlphaTuple:
    if gradient_unit_interval < 0 or gradient_unit_interval > 1:
        raise ValueError(f"{x} is out of bounds (0-1)")
    grad_color = tuple([
        clamp(lerp(gradient_unit_interval, (0, val_from), (1, val_to)), val_from, val_to)
        for val_from, val_to in zip(color_from, color_to)
    ])
    return grad_color

## Experiment ID Input

In [ ]:
experiment_ids = ["TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
detector_code = proc.Detector.ONE

In [ ]:
# default_fit_input = 2  # changed to peak finder mode, approved by Fatima 2024-07-18
# fit_input = helpers.get_input_with_default(
#     """\
# Which bimodal fit type do you want to use?
# 1: Bounds based
# 2: Peak finder based (default)
# Press Enter for default
# """,
#     default_fit_input,
#     int
# )

# fit_styles: dict[int, SliceFitStyle] = {
#     1: "bounds",
#     2: "peak_finder"
# }
# fit_style = fit_styles.get(fit_input, fit_styles[default_fit_input])
fit_style = "peak_finder"

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False

calib_params = DetectorCalibrationParams[detector_code]
lower_energy_bound = 1000 * calib_params.p1 * lower_energy_bound + calib_params.p2

settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

In [ ]:
# bin_length = helpers.get_input_with_default(
#     "Enter bin length (in seconds), or press Enter for default (300 s)",
#     300,
#     int
# )
bin_length = 300
bin_string = f"{bin_length}s"

In [ ]:
fig_4e_image_path = Path() / "Efficiency.png"

## Data Loading and Initial Processing

### Data Loading

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load_parquet_psd(exp_id)

In [ ]:
fig_4e_image = plt.imread(fig_4e_image_path)

### Initial Processing

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, detector_code)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
adc_width = 20
# adc_width = 100
# overall_settings['scan_idx'] = f"({start_scan_idx}, {end_scan_idx})"
# overall_settings['energy_width'] = energy_width

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_adc_histogram(
        psd_report,
        adc_width=adc_width,
        # psd_bin_count=50
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

## Neutron Classification

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        DetectorDataframeColumn.ENERGY,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Get experiment start time
for exp_name, data_dict in experiment_neutron_data.items():
    exp_root = get_exp_root(exp_name)
    with open(exp_root / 'exp_info.toml') as exp_info:
        exp_start_line = [line for line in exp_info if "exp_start" in line][0]
    exp_start_text = exp_start_line.replace("exp_start = ", "").strip()
    exp_start = datetime.fromisoformat(exp_start_text).astimezone(timezone.utc)
    data_dict[ExperimentDataKey.START_TIME] = exp_start

In [ ]:
# Get timetag as clock time
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
    exp_start = data_dict[ExperimentDataKey.START_TIME]

    psd_report = calculate_event_time(psd_report, exp_start)

    data_dict[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Separate neutron and gamma events
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]

    n_classify_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    neutrons_only = psd_report.query(n_classify_col_name).copy()
    gamma_only = psd_report.query(f"~{n_classify_col_name}").copy()
    data_dict[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    data_dict[ExperimentDataKey.GAMMA_ONLY] = gamma_only

## Data Binning

In [ ]:
# Create bins
for exp_name, data_dict in experiment_neutron_data.items():
    print(exp_name)
    neutron_report = data_dict[ExperimentDataKey.NEUTRONS_ONLY]

    event_time_col = DetectorDataframeColumn.EVENT_TIME.value
    start_time = neutron_report[event_time_col].min()
    end_time = neutron_report[event_time_col].max()
    print(neutron_report[event_time_col])
    timetag_clock_bins = pd.date_range(
        start=start_time, end=end_time, freq=bin_string)
    data_dict[ExperimentDataKey.TIME_BIN_EDGES] = timetag_clock_bins

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    time_bin_edges = exp_data[ExperimentDataKey.TIME_BIN_EDGES]

    time_bin_histogram_data = {}
    bin_idxs = [0, 2]
    for bin_index in bin_idxs:
        low_time_edge = time_bin_edges[bin_index]
        high_time_edge = time_bin_edges[bin_index + 1]
        event_time_col = DetectorDataframeColumn.EVENT_TIME.value
        time_bin_df = psd_report[
            psd_report[event_time_col].between(low_time_edge, high_time_edge)
        ]

        Z, xe, ye = get_psd_adc_histogram(
            time_bin_df,
            adc_width=adc_width
        )
        time_bin_histogram_data[bin_index] = {
            "histogram": Z,
            "energy_edges": xe,
            "psd_edges": ye
        }
    exp_data["time_bin_histogram"] = time_bin_histogram_data
    # Z, xe, ye = get_psd_adc_histogram(
    #     psd_report,
    #     adc_width=adc_width,
    #     # psd_bin_count=50
    # )
    # exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    # exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    # exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye

In [ ]:
# Bin neutron data
for exp_name, data_dict in experiment_neutron_data.items():
    neutron_report = data_dict[ExperimentDataKey.NEUTRONS_ONLY]
    time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]

    time_col_name = DetectorDataframeColumn.EVENT_TIME.value
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    count_col_name = BinningDataframeColumn.COUNT.value
    count_error_col_name = BinningDataframeColumn.COUNT_ERROR.value
    bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
    n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
    n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value

    start_time = time_bins[0]

    binned_neutrons = get_time_cut(
        neutron_report, time_col_name, time_bins)
    binned_neutrons = neutron_report.groupby(
        time_bin_col_name, as_index=True, observed=False) \
        .size() \
        .to_frame() \
        .copy()
    binned_neutrons.columns = [count_col_name]
    binned_neutrons[count_error_col_name] = np.sqrt(
        binned_neutrons[count_col_name]
    )

    binned_neutron_time_bins = binned_neutrons.index.to_series()
    midpoints = binned_neutron_time_bins.apply(lambda x: x.mid)
    durations = binned_neutron_time_bins.apply(
        lambda x: x.length.total_seconds()
    ).astype(np.float64)

    binned_neutrons[bin_mid_col_name] = midpoints
    binned_neutrons = bin_midpoint_time_to_seconds(binned_neutrons, start_time)

    binned_neutrons[n_rate_col_name] = (
        binned_neutrons[count_col_name] / durations)
    binned_neutrons[n_error_col_name] = (
        binned_neutrons[count_error_col_name] / durations)
    binned_neutrons = binned_neutrons.drop(
        [count_col_name, count_error_col_name],
        axis=1
    ) \
        .copy()
    data_dict[ExperimentDataKey.BINNED_NEUTRONS] = binned_neutrons

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

In [ ]:
chosen_slice_idx = 12

In [ ]:
def plot_figure_4a(ax: mpl.axes.Axes):
    cmap = plt.colormaps["viridis"]
    fontsize_offset = 0
    # histo_res = 128
    # contour_res = 100
    angle_elev = 30
    angle_rot = -20
    e_margin = 2
    psd_margin = 0.002

    exp_name = "TB-26"
    data_dict = experiment_neutron_data[exp_name]
    xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    dz = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
    # borders = data_dict[ExperimentDataKey.BORDERS]

    x, y = np.meshgrid(xe[:-1], ye[:-1])
    x, y = x.ravel(), y.ravel()
    xi, yi = np.meshgrid(range(len(xe)-1), range(len(ye)-1))
    xi, yi = xi.ravel(), yi.ravel()
    z = np.full_like(x, 0)
    _dx = xe[1:] - xe[:-1]
    _dy = ye[1:] - ye[:-1]
    dx, dy = np.meshgrid(_dx, _dy)
    dz = dz.T
    dx, dy, dz = dx.ravel(), dy.ravel(), dz.ravel()

    x = x + e_margin
    dx = dx - e_margin
    y = y + psd_margin
    dy = dy - psd_margin

    height_mask = dz > 10
    x = x[height_mask]
    y = y[height_mask]
    xi = xi[height_mask]
    yi = yi[height_mask]
    z = z[height_mask]
    dx = dx[height_mask]
    dy = dy[height_mask]
    dz = dz[height_mask]
    # within_borders = within_borders[height_mask]

    min_dz = -2000
    max_dz = np.max(dz)
    norm = mpl.colors.Normalize(vmin=min_dz, vmax=max_dz)
    mapped_colors = [cmap(norm(dz_val)) for dz_val in dz]
    is_chosen_slice = xi == chosen_slice_idx
    mapped_colors = [
        color if is_chosen else (color[0], color[1], color[2], 0.02)
        for is_chosen, color
        in zip(is_chosen_slice, mapped_colors)
    ]
    # mapped_lws = [0.2 if is_chosen else 0 for is_chosen in is_chosen_slice]
    # mapped_ecs = ["black" if is_chosen else (0, 0, 0, 0) for is_chosen in is_chosen_slice]
    # mapped_colors = [
    #     bg_red if is_within_borders else color
    #     for is_within_borders, color in zip(within_borders, mapped_colors)
    # ]

    ax.view_init(angle_elev, angle_rot)
    ax.bar3d(
        x, y, z, dx, dy, dz,
        # color=color_alphas
        color=mapped_colors,
        shade=False,
        zsort="max",
        # lw=0.2,
        # lw=mapped_lws,
        # ec="black"
        # ec=mapped_ecs
    )

    window_x = xe[chosen_slice_idx]
    bg_grey_alpha = to_color_alpha_tuple(bg_grey)
    bg_grey_alpha = (bg_grey_alpha[0], bg_grey_alpha[1], bg_grey_alpha[2], 0.2)
    window = mpl.patches.Rectangle((0, 0), 0.5, 4000, fc=bg_grey_alpha, ec="black", lw=3, zorder=0)
    # window_frame = mpl.patches.Rectangle((0, 0), 0.5, 4000, fc=None, lw=3, ec="black", zorder=1)
    ax.add_patch(window)
    # ax.add_patch(window_frame)
    art3d.pathpatch_2d_to_3d(window, z=window_x, zdir="x")
    # art3d.pathpatch_2d_to_3d(window_frame, z=window_x, zdir="x")

    # ax.set_title(f"{exp_name} PSD/Energy 3D Histogram", fontsize=fontsize+4)
    ax.set_xlabel("Energy\n(ADC channel x1000)", fontsize=fontsize+fontsize_offset)
    ax.set_ylabel("PSD", fontsize=fontsize+fontsize_offset)
    ax.set_zlabel("Counts (x1000)", fontsize=fontsize+fontsize_offset)
    ax.set_xlim(0, 3000)
    ax.set_ylim(0, 0.5)
    ax.set_zlim(0, 4000)
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x / 1000:.1f}")
    ax.zaxis.set_major_formatter(lambda z, pos: f"{z / 1000:.1f}")
    ax.tick_params(labelsize=fontsize+fontsize_offset)
    ax.tick_params(pad=0)
    ax.tick_params(axis="x", pad=8)
    ax.tick_params(axis="y", pad=4)
    # ax.tick_params(axis="z", pad=-2)
    # for axis3d in [ax.xaxis, ax.yaxis, ax.zaxis]:
    #     axis3d.labelpad = 40
    ax.xaxis.labelpad = 2.000 * fontsize
    ax.yaxis.labelpad = 1.250 * fontsize
    ax.zaxis.labelpad = 1.375 * fontsize
    xaxis_ticklabels = ax.xaxis.get_ticklabels()
    for ticklabel in xaxis_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("baseline")
    yaxis_ticklabels = ax.yaxis.get_ticklabels()
    for ticklabel in yaxis_ticklabels:
        ticklabel.set_ha("center")
        ticklabel.set_va("top")
    zaxis_ticklabels = ax.zaxis.get_ticklabels()
    for ticklabel in zaxis_ticklabels:
        ticklabel.set_ha("left")
        ticklabel.set_va("center_baseline")
    ax.xaxis.set_pane_color((1, 1, 1, 0))
    ax.yaxis.set_pane_color((1, 1, 1, 0))
    ax.zaxis.set_pane_color((1, 1, 1, 0))
    # ax.set_box_aspect(None, zoom=0.80)

In [ ]:
def plot_figure_4b(ax: mpl.axes.Axes):
    exp_data = experiment_neutron_data["TB-26"]
    fom_df = exp_data[ExperimentDataKey.FOM_RESULTS]
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    # xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]

    fom_slice = fom_df.iloc[chosen_slice_idx]
    mu1, sigma1, a1 = [fom_slice[val_name] for val_name in ["mu1", "sigma1", "a1"]]
    mu2, sigma2, a2 = [fom_slice[val_name] for val_name in ["mu2", "sigma2", "a2"]]
    Z_slice = Z[chosen_slice_idx, :]
    ymids = (ye[:-1] + ye[1:]) / 2

    gauss_y = np.linspace(0, 0.6, 1000)
    gamma_gaussian = proc.gaussian(gauss_y, mu1, sigma1, a1)
    neutron_gaussian = proc.gaussian(gauss_y, mu2, sigma2, a2)
    gamma_vline_xs = [mu1, mu1 + 5 * sigma1]
    neutron_vline_xs = [mu2, mu2 + 5 * sigma2]
    # gamma_vline_labels = [r"${\mu}_{\gamma}$", r"${\mu}_{\gamma} + 5{\sigma}_{\gamma}$"]
    # neutron_vline_labels = [r"${\mu}_{n}$", r"${\mu}_{n} + 5{\sigma}_{n}$"]
    # gamma_ymax = [proc.gaussian(x, mu1, sigma1, a1) for x in gamma_vline_xs]
    # neutron_ymax = [proc.gaussian(x, mu2, sigma2, a2) for x in neutron_vline_xs]
    # print(gamma_ymax)
    # print(gamma_ymax + neutron_ymax)

    ax.plot(gauss_y, gamma_gaussian, lw=5, color=bg_grey)
    ax.plot(gauss_y, neutron_gaussian, lw=5, color=bg_grey)
    ax.plot(ymids, Z_slice, "-", lw=2, color="black", alpha=0.5)

    # trans = ax.transData + ax.transAxes.inverted()
    # trans = mpl.transforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.axvline(0, 0, 3000, alpha=0)  # "burner" line - first line never transforms properly (transform not initiated?)
    for vline_x in gamma_vline_xs + neutron_vline_xs:
        ax.axvline(vline_x, 0, 1, lw=2, color="black")
        # ax.text(vline_x + 0.005, 3100, vline_label, fontsize=fontsize-10)
    margin = 0.005
    ax.text(
        mu1 - margin,
        3100,
        r"${\mu}_{\gamma}$",
        fontsize=fontsize-5,
        ha="right"
    )
    ax.text(
        (mu1 + 5 * sigma1) - margin,
        3100,
        r"${\mu}_{\gamma} + 5{\sigma}_{\gamma}$",
        fontsize=fontsize-5,
        ha="right"
    )
    ax.text(
        mu2 - margin,
        3100,
        r"${\mu}_{n}$",
        fontsize=fontsize-5,
        ha="right"
    )
    ax.text(
        (mu2 + 5 * sigma2) - margin,
        3100,
        r"${\mu}_{n} + 5{\sigma}_{n}$",
        fontsize=fontsize-5,
        ha="right"
    )

    neutron_region = (gauss_y >= (mu1 + 5 * sigma1)) & (gauss_y <= (mu2 + 5 * sigma2))
    ax.fill_between(gauss_y, neutron_gaussian, where=neutron_region, color=bg_red)
    ax.fill_between(gauss_y, neutron_gaussian, where=~neutron_region, color=bg_bluegrey)
    ax.fill_between(gauss_y, gamma_gaussian, color=bg_bluegrey)

    ax.set_xlabel("PSD", fontsize=fontsize)
    ax.set_ylabel("Counts (x1000)", fontsize=fontsize)
    ax.set_xlim(0, 0.55)
    ax.set_ylim(0, 3300)
    ax.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
    ax.xaxis.set_major_formatter(lambda x, _: "" if x == 0 else f"{x:.1f}")
    ax.tick_params(labelsize=fontsize)

In [ ]:
def plot_figure_4c(ax: mpl.axes.Axes):
    cmap = plt.colormaps["viridis"]
    # figsize = (24, 24)
    # fontsize = 32
    fontsize_offset = 0
    # histo_res = 128
    # contour_res = 100
    angle_elev = 30
    angle_rot = -20
    e_margin = 2
    psd_margin = 0.002

    data_dict = experiment_neutron_data["TB-26"]
    xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    dz = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
    borders = data_dict[ExperimentDataKey.BORDERS]

    x, y = np.meshgrid(xe[:-1], ye[:-1])
    x, y = x.ravel(), y.ravel()
    z = np.full_like(x, 0)
    _dx = xe[1:] - xe[:-1]
    _dy = ye[1:] - ye[:-1]
    dx, dy = np.meshgrid(_dx, _dy)
    dz = dz.T
    dx, dy, dz = dx.ravel(), dy.ravel(), dz.ravel()

    _xmids = (xe[:-1] + xe[1:]) / 2
    _ymids = (ye[:-1] + ye[1:]) / 2
    xmids, ymids = np.meshgrid(_xmids, _ymids)
    xmids, ymids = xmids.ravel(), ymids.ravel()
    if borders.left is None:
        within_left_border = np.full_like(xmids, True, dtype=bool)
    else:
        within_left_border = xmids >= borders.left
    if borders.right is None:
        within_right_border = np.full_like(xmids, True, dtype=bool)
    else:
        within_right_border = xmids <= borders.right
    if borders.bottom is None:
        within_bottom_border = np.full_like(ymids, True, dtype=bool)
    else:
        bottom_border_psds = borders.bottom(xmids)
        within_bottom_border = ymids >= bottom_border_psds
    if borders.top is None:
        within_top_border = np.full_like(ymids, True, dtype=bool)
    else:
        top_border_psds = borders.top(xmids)
        within_top_border = ymids <= top_border_psds
    within_borders = (within_left_border &
                      within_right_border &
                      within_bottom_border &
                      within_top_border)

    x = x + e_margin
    dx = dx - e_margin
    y = y + psd_margin
    dy = dy - psd_margin
    
    height_mask = dz > 10
    x = x[height_mask]
    y = y[height_mask]
    z = z[height_mask]
    dx = dx[height_mask]
    dy = dy[height_mask]
    dz = dz[height_mask]
    within_borders = within_borders[height_mask]

    min_dz = -2000
    max_dz = np.max(dz)
    norm = mpl.colors.Normalize(vmin=min_dz, vmax=max_dz)
    mapped_colors = [cmap(norm(dz_val)) for dz_val in dz]
    mapped_colors = [
        bg_red if is_within_borders else color
        for is_within_borders, color in zip(within_borders, mapped_colors)
    ]

    ax.view_init(angle_elev, angle_rot)
    ax.bar3d(
        x, y, z, dx, dy, dz,
        # color=color_alphas
        color=mapped_colors,
        shade=False,
        zsort="max",
        lw=0.2,
        ec="black"
    )

    border_e = np.linspace(lower_energy_bound, 3000, 200, dtype="float")
    bottom_border_psds = borders.bottom(border_e)
    top_border_psds = borders.top(border_e)
    border_zs = np.full_like(border_e, 0.000001, dtype="float")
    ax.plot(border_e, bottom_border_psds, zs=0, zdir='z', lw=3, color="red", zorder=0)
    ax.plot(border_e, top_border_psds, zs=0, zdir='z', lw=3, color="red", zorder=0)
    ax.fill_between(
        border_e, bottom_border_psds, border_zs,
        border_e, top_border_psds, border_zs,
        color="red", alpha=0.5, zorder=0
    )

    ax.set_xlabel("Energy\n(ADC channel x1000)", fontsize=fontsize+fontsize_offset)
    ax.set_ylabel("PSD", fontsize=fontsize+fontsize_offset)
    ax.set_zlabel("Counts (x1000)", fontsize=fontsize+fontsize_offset)
    ax.set_xlim(0, 3000)
    ax.set_ylim(0, 0.5)
    ax.set_zlim(0, 4000)
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x / 1000:.1f}")
    ax.zaxis.set_major_formatter(lambda z, pos: f"{z / 1000:.1f}")
    ax.tick_params(labelsize=fontsize+fontsize_offset)
    ax.tick_params(pad=0)
    ax.tick_params(axis="x", pad=8)
    ax.tick_params(axis="y", pad=4)
    # ax.tick_params(axis="z", pad=-2)
    # for axis3d in [ax.xaxis, ax.yaxis, ax.zaxis]:
    #     axis3d.labelpad = 40
    ax.xaxis.labelpad = 2.000 * fontsize
    ax.yaxis.labelpad = 1.250 * fontsize
    ax.zaxis.labelpad = 1.375 * fontsize
    xaxis_ticklabels = ax.xaxis.get_ticklabels()
    for ticklabel in xaxis_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("baseline")
    yaxis_ticklabels = ax.yaxis.get_ticklabels()
    for ticklabel in yaxis_ticklabels:
        ticklabel.set_ha("center")
        ticklabel.set_va("top")
    zaxis_ticklabels = ax.zaxis.get_ticklabels()
    for ticklabel in zaxis_ticklabels:
        ticklabel.set_ha("left")
        ticklabel.set_va("center_baseline")
    ax.xaxis.set_pane_color((1, 1, 1, 0))
    ax.yaxis.set_pane_color((1, 1, 1, 0))
    ax.zaxis.set_pane_color((1, 1, 1, 0))
    ax.set_box_aspect(None, zoom=0.85)

In [ ]:
def plot_figure_4d(ax: mpl.axes.Axes):
    data_dict = experiment_neutron_data["TB-26"]
    # dot_size = 8

    binned_neutrons = data_dict[ExperimentDataKey.BINNED_NEUTRONS]

    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
    # n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value
    zeroed_bins = binned_neutrons[bin_time_col_name] / 60
    rates = binned_neutrons[n_rate_col_name]
    # rate_errors = binned_neutrons[n_error_col_name]
    # print(zeroed_bins[:5])
    # print(rates[:5])

    ax.errorbar(
        zeroed_bins,
        rates,
        # yerr=rate_errors,
        # fmt=".",
        linestyle=':',
        # markersize=dot_size,
        # capsize=dot_size,
        markersize=0,
        color="black"
    )
    ax.bar(
        zeroed_bins,
        rates,
        # width=(zeroed_bins[1:] - zeroed_bins[:-1]),
        bin_length / 60,
        linewidth=1,
        edgecolor="black",
        facecolor="#00000000"
    )
    ax.set_xlabel("Time [minutes]", fontsize=fontsize)  # Update x-axis label
    ax.set_ylabel("Neutron count rate [1/s]", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    ax.set_xlim(0, 130)
    ax.set_ylim(0, 300)

In [ ]:
def plot_figure_4d_side(ax: mpl.axes.Axes, bin_idx: int):
    cmap = plt.colormaps["viridis"]
    figsize = (24, 24)
    fontsize = 16
    # histo_res = 128
    # contour_res = 100
    angle_elev = 30
    angle_rot = -20
    e_margin = 2
    psd_margin = 0.002

    exp_data = experiment_neutron_data["TB-26"]
    histo_data = exp_data["time_bin_histogram"][bin_idx]
    # xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
    # ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    # dz = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
    xe = histo_data["energy_edges"]
    ye = histo_data["psd_edges"]
    dz = histo_data["histogram"]
    
    x, y = np.meshgrid(xe[:-1], ye[:-1])
    x, y = x.ravel(), y.ravel()
    z = np.full_like(x, 0)
    _dx = xe[1:] - xe[:-1]
    _dy = ye[1:] - ye[:-1]
    dx, dy = np.meshgrid(_dx, _dy)
    dz = dz.T
    dx, dy, dz = dx.ravel(), dy.ravel(), dz.ravel()

    _xmids = (xe[:-1] + xe[1:]) / 2
    _ymids = (ye[:-1] + ye[1:]) / 2
    xmids, ymids = np.meshgrid(_xmids, _ymids)
    xmids, ymids = xmids.ravel(), ymids.ravel()
    # xmids = recalibrate_np(xmids)
    # ymids = recalibrate_np(ymids)
    if borders.left is None:
        within_left_border = np.full_like(xmids, True, dtype=bool)
    else:
        within_left_border = xmids >= borders.left
    if borders.right is None:
        within_right_border = np.full_like(xmids, True, dtype=bool)
    else:
        within_right_border = xmids <= borders.right
    if borders.bottom is None:
        within_bottom_border = np.full_like(ymids, True, dtype=bool)
    else:
        bottom_border_psds = borders.bottom(xmids)
        within_bottom_border = ymids >= bottom_border_psds
    if borders.top is None:
        within_top_border = np.full_like(ymids, True, dtype=bool)
    else:
        top_border_psds = borders.top(xmids)
        within_top_border = ymids <= top_border_psds
    within_borders = (within_left_border &
                      within_right_border &
                      within_bottom_border &
                      within_top_border)

    x = x + e_margin
    dx = dx - e_margin
    y = y + psd_margin
    dy = dy - psd_margin
    
    height_mask = dz > 1
    x = x[height_mask]
    y = y[height_mask]
    z = z[height_mask]
    dx = dx[height_mask]
    dy = dy[height_mask]
    dz = dz[height_mask]
    within_borders = within_borders[height_mask]

    # min_dz = np.min(dz)
    # min_dz = -2000
    min_dz = 0
    max_dz = np.max(dz)
    norm = mpl.colors.Normalize(vmin=min_dz, vmax=max_dz)
    mapped_colors = [cmap(norm(dz_val)) for dz_val in dz]
    mapped_colors = [
        bg_red if is_within_borders else color
        for is_within_borders, color in zip(within_borders, mapped_colors)
    ]

    ax.view_init(angle_elev, angle_rot)
    ax.bar3d(x, y, z, dx, dy, dz,
             # color=color_alphas
             color=mapped_colors,
             shade=False,
             zsort="max",
             lw=0.2,
             ec="black"
            )

    border_e = np.linspace(lower_energy_bound, 3000, 200, dtype="float")
    bottom_border_psds = borders.bottom(border_e)
    top_border_psds = borders.top(border_e)
    border_zs = np.zeros_like(border_e, dtype="float")
    ax.plot(
        border_e,
        bottom_border_psds,
        zs=0,
        zdir='z',
        axlim_clip=True,
        lw=3,
        color="red",
        zorder=0
    )
    ax.plot(
        border_e,
        top_border_psds,
        zs=0,
        zdir='z',
        axlim_clip=True,
        lw=3,
        color="red",
        zorder=0
    )
    ax.fill_between(
        border_e, bottom_border_psds, border_zs,
        border_e, top_border_psds, border_zs,
        axlim_clip=True, color="red", alpha=0.5, zorder=0
    )
    
    # bbox_params = {
    #     # "boxstyle": "round, pad=0.003, rounding_size=0.1",
    #     "fc": "white",
    #     "lw": 1,
    #     "fill": True,
    #     "alpha": 0.9
    # }
    # text_params = {
    #     "fontsize": fontsize-2,
    #     "bbox": bbox_params,
    #     "zorder": 5,
    #     "va": "center"
    # }
    
    # gamma_text = ["Gamma", "channel"]
    # neutron_text = ["Neutron", "channel"]
    # # scaling = (400, 0.025)
    # scaling = (0.03, 280)
    # gamma_position = (4400, 0.135)
    # neutron_position = (4400, 0.35)
    # # mutation_aspect = 3500 / 0.5
    # # mutation_aspect = None
    # linespacing = 1
    # padding = 0.003
    # rounding_size = 0.1
    # font_properties = mpl.font_manager.FontProperties(weight="bold")

    # for text_list, position in [
    #     (gamma_text, gamma_position),
    #     (neutron_text, neutron_position)
    # ]:
    #     text_path = make_text_path(
    #         text_list,
    #         linespacing,
    #         # font_properties=font_properties
    #     )
    #     # bbox = make_bounding_box_for_text_path(
    #     #     text_path,
    #     #     padding,
    #     #     rounding_size,
    #     #     1,
    #     #     # mutation_aspect=mutation_aspect,
    #     #     **bbox_params
    #     # )
    #     transform = mpl.transforms.Affine2D()
    #     transform = transform.scale(*scaling)
    #     pos_from = get_bbox_center(text_path.get_extents(transform))
    #     x_translate, y_translate = get_translation_to(pos_from, position)
    #     transform = transform.translate(x_translate, y_translate)
    #     x_center, y_center = get_bbox_center(text_path.get_extents(transform))
    #     transform = transform.rotate_deg_around(x_center, y_center, 90)
    #     text_path = transform.transform_path(text_path)
    #     text_patch = convert_text_path_to_patch(text_path, 1)
    #     # patch_to_3d_plot_wall(bbox, ax, "z")
    #     patch_to_3d_plot_wall(text_patch, ax, "z")

    # arrow_width = 0.04
    # arrow_head_width = 0.07
    # arrow_head_length = 400
    # arrow_base_e = 3950
    # arrow_len_psd = 0
    # arrow_kwargs = {
    #     "width": arrow_width,
    #     "head_width": arrow_head_width,
    #     "head_length": arrow_head_length,
    #     "length_includes_head": True,
    #     "ec": "black",
    #     "fc": "none",
    #     "lw": 4
    # }
    # arrow_g = mpl.patches.FancyArrow(
    #     arrow_base_e, 0.135, -800, arrow_len_psd,
    #     **arrow_kwargs
    # )
    # patch_to_3d_plot_wall(arrow_g, ax, "z")
    # arrow_n = mpl.patches.FancyArrow(
    #     arrow_base_e, 0.35, -2550, arrow_len_psd,
    #     **arrow_kwargs
    # )
    # patch_to_3d_plot_wall(arrow_n, ax, "z")

    # ax.set_title(f"{exp_name} PSD/Energy 3D Histogram", fontsize=fontsize+4)
    # ax.set_xlabel("Energy (ADC channel x1000)", fontsize=fontsize)
    # ax.set_ylabel("PSD", fontsize=fontsize)
    # ax.set_zlabel("Counts", fontsize=fontsize)
    ax.set_xlim(0, 3000)
    ax.set_ylim(0, 0.5)
    ax.set_zlim(0, 200)
    # ax.xaxis.set_major_formatter(lambda x, pos: f"{x / 1000:.1f}")
    # ax.tick_params(labelsize=fontsize)
    # ax.tick_params(pad=0)
    # ax.tick_params(axis="x", pad=8)
    # ax.tick_params(axis="y", pad=4)
    ax.tick_params(labelleft=False, labelright=False, labelbottom=False, labeltop=False)
    ax.xaxis.labelpad = 24
    ax.yaxis.labelpad = 20
    ax.zaxis.labelpad = 22
    xaxis_ticklabels = ax.xaxis.get_ticklabels()
    for ticklabel in xaxis_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("baseline")
    yaxis_ticklabels = ax.yaxis.get_ticklabels()
    for ticklabel in yaxis_ticklabels:
        ticklabel.set_ha("center")
        ticklabel.set_va("top")
    zaxis_ticklabels = ax.zaxis.get_ticklabels()
    for ticklabel in zaxis_ticklabels:
        ticklabel.set_ha("left")
        ticklabel.set_va("center_baseline")
    ax.xaxis.set_pane_color((1, 1, 1, 0))
    ax.yaxis.set_pane_color((1, 1, 1, 0))
    ax.zaxis.set_pane_color((1, 1, 1, 0))
    ax.set_box_aspect(None, zoom=0.85)

In [ ]:
def plot_figure_4e(ax: mpl.axes.Axes):
    ax.imshow(fig_4e_image, aspect="equal")
    ax.set_xlim(500, 5000)
    ax.set_ylim(2250, 700)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine_key in ["left", "right", "bottom", "top"]:
        ax.spines[spine_key].set_visible(False)

In [ ]:
dl_folder = Path.home() / "Downloads"
mosaic = """
ABB
C..
CDD
EEE
"""
fig, ax_dict = plt.subplot_mosaic(
    mosaic,
    figsize=(17, 22),
    height_ratios=[2, 1, 1, 2],
    width_ratios=[2, 1, 1],
    subplot_kw={
        "xmargin": 0,
        "ymargin": 0
    },
    # per_subplot_kw={
    #     # "ACXY": {"projection": "3d", "computed_zorder": False},
    #     # "D": {"ymargin": 20}
    # },
    # dpi=600,
    layout="constrained",
    gridspec_kw={
        "wspace": 0.001,
        "hspace": 0.001
    },
)

# axA = ax_dict["A"]
axB = ax_dict["B"]
axD = ax_dict["D"]
# axX = ax_dict["X"]
# axY = ax_dict["Y"]
# plot_figure_4a(axA)
plot_figure_4b(axB)
# plot_figure_4c(ax_dict["C"])
plot_figure_4d(axD)
# plot_figure_4d_side(axX, 0)
# plot_figure_4d_side(axY, 2)
plot_figure_4e(ax_dict["E"])

# conA = mpl.patches.ConnectionPatch(
#     (0.688, 0.75),
#     (0, 0.70),
#     axA.transAxes,
#     axB.transAxes,
#     axesA=axA,
#     axesB=axB,
#     lw=3,
#     color="black"
# )
# axX_ypos = 0.102
# conBX = mpl.patches.ConnectionPatch(
#     (2.5, 1.87),
#     (0.15, axX_ypos),
#     axD.transData,
#     axX.transAxes,
#     axesA=axD,
#     axesB=axX,
#     lw=3,
#     color="black"
# )
# conBY = mpl.patches.ConnectionPatch(
#     (12.5, 197.4),
#     (0.50, axX_ypos),
#     axD.transData,
#     axY.transAxes,
#     axesA=axD,
#     axesB=axY,
#     lw=3,
#     color="black"
# )
# fig.add_artist(conA)
# fig.add_artist(conBX)
# fig.add_artist(conBY)

# plt.tight_layout(pad=1.01)
# fig.savefig(dl_folder / "fig4.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
input("Processing done, hit Enter to finish")
stop()

In [ ]:
# Fig. 4a
fig, ax = plt.subplots(
    1, 1,
    figsize=(8.5, 7.3),
    # dpi=600,
    # frameon=False,
    subplot_kw={"projection": "3d"},
    layout="constrained",
)
plot_figure_4a(ax)
ax.set_box_aspect(None, zoom=0.8)
# fig.savefig(dl_folder / "fig4a.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
# Fig. 4b
fig, ax = plt.subplots(
    1, 1,
    figsize=(8.5, 7.3),
    # dpi=600,
    # frameon=False,
    # subplot_kw={"projection": "3d"},
    layout="constrained",
)
plot_figure_4b(ax)
# ax.set_box_aspect(None, zoom=0.8)
# fig.savefig(dl_folder / "fig4b.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
# Fig. 4c
fig, ax = plt.subplots(
    1, 1,
    figsize=(8.5, 7.3),
    # dpi=600,
    # frameon=False,
    subplot_kw={"projection": "3d"},
    layout="constrained",
)
plot_figure_4c(ax)
ax.set_box_aspect(None, zoom=0.8)
# fig.savefig(dl_folder / "fig4c.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
# Fig. 4d_main
fig, ax = plt.subplots(
    1, 1,
    figsize=(8.5, 3.65),
    # dpi=600,
    # frameon=False,
    # subplot_kw={"projection": "3d"},
    layout="constrained",
)
plot_figure_4d(ax)
# ax.set_box_aspect(None, zoom=0.8)
# fig.savefig(dl_folder / "fig4d_main.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
# Fig. 4d_upper_left

In [ ]:
# Fig. 4d_upper_right